In [ ]:
import pandas as pd
import numpy as np
import joblib
import itertools
import warnings
import os
from dotenv import load_dotenv
from supabase import create_client
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import log_loss, brier_score_loss
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

load_dotenv()
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')
if SUPABASE_KEY is None or SUPABASE_URL is None:
    print('URL or key did not load properly')
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
def fetch_all(table):
    rows = []
    chunk = 1000
    offset = 0
    while True:
        result = supabase.table(table).select('*').range(offset, offset + chunk - 1).execute()
        rows.extend(result.data)
        if len(result.data) < chunk:
            break
        offset += chunk
    return pd.DataFrame(rows)

df = fetch_all('shots')
print(df.shape)
df.head()

In [ ]:
backup = df.copy()

In [ ]:
df = backup.copy()

df = df[df['result'] != 'OwnGoal'].copy()
df = df[df['situation'] != 'Penalty'].copy()
df['is_goal'] = (df['result'] == 'Goal').astype(int)
df['understat_xg'] = df['understat_xg'].astype(float)
df['x'] = df['x'].astype(float)
df['y'] = df['y'].astype(float)
df['minute'] = df['minute'].astype(int)
df['season'] = df['season'].astype(int)

df['player_assisted'] = df['player_assisted'].str.replace('&#039;', "'", regex=False)
df['is_assisted'] = df['player_assisted'].notnull().astype(int)

df = df.drop(columns=['model_xg', 'player', 'player_id', 'h_team', 'a_team', 'date', 'match_id', 'player_assisted'])

noise_actions = ['SubstitutionOn', 'SubstitutionOff', 'FormationChange', 'Start', 'End']
df['last_action'] = df['last_action'].replace(noise_actions, 'Other')

df['game_phase'] = pd.cut(
    df['minute'],
    bins=[0, 15, 30, 45, 60, 75, 90, 120],
    labels=['0-15', '15-30', '30-45', '45-60', '60-75', '75-90', '90+']
)

df = pd.get_dummies(df, columns=['last_action', 'situation', 'shot_type', 'league', 'h_a', 'game_phase'])

GOAL_X, GOAL_Y = 1.0, 0.5
df['distance'] = np.sqrt((df['x'] - GOAL_X)**2 + (df['y'] - GOAL_Y)**2)
df['distance_squared'] = df['distance'] ** 2
df['off_central'] = abs(df['y'] - 0.5).astype(float)

def goal_angle(x, y):
    left  = np.array([1.0, 0.458])
    right = np.array([1.0, 0.542])
    a = np.array([x, y])
    ba = a - left
    ca = a - right
    cosine = np.dot(ba, ca) / (np.linalg.norm(ba) * np.linalg.norm(ca) + 1e-6)
    return np.arccos(np.clip(cosine, -1, 1))

df['goal_angle'] = df.apply(lambda r: goal_angle(r['x'], r['y']), axis=1)
df['distance_x_angle'] = df['distance'] * df['goal_angle']

df['in_six_yard_box'] = ((df['x'] > 0.88) & (df['y'].between(0.35, 0.65))).astype(int)
df['in_penalty_box']  = ((df['x'] > 0.70) & (df['y'].between(0.2,  0.8))).astype(int)

print(df.shape)
df.head()

In [ ]:
baseline_logloss = log_loss(df['is_goal'], df['understat_xg'])
baseline_brier   = brier_score_loss(df['is_goal'], df['understat_xg'])
print(f'Understat log-loss:    {baseline_logloss:.4f}')
print(f'Understat Brier score: {baseline_brier:.4f}')

fraction_pos, mean_pred = calibration_curve(df['is_goal'], df['understat_xg'], n_bins=20)
plt.plot(mean_pred, fraction_pos, marker='o', label='Understat')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect')
plt.xlabel('Predicted xG')
plt.ylabel('Actual goal rate')
plt.title('Understat Calibration Curve')
plt.legend()
plt.show()

In [ ]:
# Use most recent 3 seasons as test, everything else as train
# Within each season, shuffle so we get a representative sample
# This gives us more recent data without leaking the future

TEST_SEASONS = [2024, 2025]

drop_cols = ['id', 'result', 'understat_xg', 'is_goal', 'season']

train = df[~df['season'].isin(TEST_SEASONS)].copy()
test  = df[df['season'].isin(TEST_SEASONS)].copy()

# Weight recent seasons more heavily in training
max_season = train['season'].max()
train['weight'] = train['season'].apply(lambda s: 1 + (s - 2014) / (max_season - 2014))
sample_weights = train['weight'].values
train = train.drop(columns=['weight'])

X_train = train.drop(columns=drop_cols)
y_train = train['is_goal']
X_test  = test.drop(columns=drop_cols)
y_test  = test['is_goal']

understat_ll = log_loss(y_test, test['understat_xg'])
understat_bs = brier_score_loss(y_test, test['understat_xg'])
print(f'Test set size: {len(X_test)}')
print(f'Understat  log-loss: {understat_ll:.4f}  brier: {understat_bs:.4f}')

In [ ]:
naive = LogisticRegression()
naive.fit(X_train[['distance', 'goal_angle']], y_train)
naive_preds = naive.predict_proba(X_test[['distance', 'goal_angle']])[:, 1]
naive_ll = log_loss(y_test, naive_preds)
print(f'Naive baseline: {naive_ll:.4f}')
print(f'Understat:      {understat_ll:.4f}')

In [ ]:
# Two-phase tuning:
# Phase 1 — coarse grid to find best region
# Phase 2 — fine grid around best from phase 1

def run_grid(param_grid, X_train, y_train, X_test, y_test, sample_weights=None):
    keys, values = zip(*param_grid.items())
    candidates = [dict(zip(keys, v)) for v in itertools.product(*values)]
    best_ll = float('inf')
    best_model = None
    best_params = None
    for params in candidates:
        m = CalibratedClassifierCV(
            XGBClassifier(
                **params,
                colsample_bytree=0.8,
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=42
            ), cv=3, method='isotonic'
        )
        if sample_weights is not None:
            m.fit(X_train, y_train, sample_weight=sample_weights)
        else:
            m.fit(X_train, y_train)
        preds = m.predict_proba(X_test)[:, 1]
        ll = log_loss(y_test, preds)
        bs = brier_score_loss(y_test, preds)
        print(f'{params} -> log-loss: {ll:.4f}  brier: {bs:.4f}')
        if ll < best_ll:
            best_ll = ll
            best_model = m
            best_params = params
    return best_model, best_params, best_ll

print('Phase 1 — coarse grid')
coarse_grid = {
    'n_estimators':  [300, 500],
    'max_depth':     [3, 4, 5],
    'learning_rate': [0.05, 0.10, 0.15],
    'subsample':     [0.8],
}
_, best_coarse, _ = run_grid(coarse_grid, X_train, y_train, X_test, y_test, sample_weights)
print(f'\nBest coarse params: {best_coarse}')

In [ ]:
print('Phase 2 — fine grid around best coarse params')

lr = best_coarse['learning_rate']
md = best_coarse['max_depth']
ne = best_coarse['n_estimators']

fine_grid = {
    'n_estimators':  [ne, ne + 100],
    'max_depth':     [max(2, md - 1), md, md + 1],
    'learning_rate': [round(lr - 0.02, 3), lr, round(lr + 0.02, 3)],
    'subsample':     [0.7, 0.8, 0.9],
}

best_model, best_params, best_ll = run_grid(fine_grid, X_train, y_train, X_test, y_test, sample_weights)
best_preds = best_model.predict_proba(X_test)[:, 1]
best_bs = brier_score_loss(y_test, best_preds)

print(f'\nBest params:            {best_params}')
print(f'Naive baseline:         {naive_ll:.4f}')
print(f'Best model log-loss:    {best_ll:.4f}  brier: {best_bs:.4f}')
print(f'Understat:              {understat_ll:.4f}')
print(f'Improvement over naive: {(naive_ll - best_ll) / naive_ll * 100:.1f}%')
print(f'Gap to Understat:       {(best_ll - understat_ll) / understat_ll * 100:.1f}%')

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect')

frac, mean = calibration_curve(y_test, best_preds, n_bins=10)
plt.plot(mean, frac, marker='o', label='Your model')

frac_u, mean_u = calibration_curve(y_test, test['understat_xg'], n_bins=10)
plt.plot(mean_u, frac_u, marker='s', label='Understat')

frac_n, mean_n = calibration_curve(y_test, naive_preds, n_bins=10)
plt.plot(mean_n, frac_n, marker='^', label='Naive baseline')

plt.xlabel('Predicted xG')
plt.ylabel('Actual goal rate')
plt.title('Calibration Curves')
plt.legend()
plt.show()

In [ ]:
model = best_model
joblib.dump(model, 'xg_model.pkl')
print(f'Saved xg_model.pkl')
print(f'Params: {best_params}')

In [ ]:
from scipy.stats import binned_statistic_2d

bin_size = 0.02
stat, x_edges, y_edges, _ = binned_statistic_2d(
    df['x'], df['y'], df['is_goal'],
    statistic='mean',
    bins=[np.arange(0, 1 + bin_size, bin_size),
          np.arange(0, 1 + bin_size, bin_size)]
)

count, _, _, _ = binned_statistic_2d(
    df['x'], df['y'], df['is_goal'],
    statistic='count',
    bins=[np.arange(0, 1 + bin_size, bin_size),
          np.arange(0, 1 + bin_size, bin_size)]
)
stat[count < 20] = np.nan

plt.figure(figsize=(8, 6))
plt.imshow(stat.T, origin='lower', extent=[0, 1, 0, 1],
           aspect='auto', cmap='hot', vmin=0, vmax=0.4)
plt.colorbar(label='Goal conversion rate')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Actual goal conversion rate by pitch location')
plt.show()